# 16.5 生产级安全护栏 (Guardrails)

> 🕐 预估学习时间：35分钟

生产系统通常在模型外挂多层护栏：输入分类、输出审查、工具权限、策略引擎（如 Llama Guard、NeMo Guardrails、正则/规则）。护栏与对齐微调互补，可热更新。

本节涵盖：
- 分层护栏架构
- 输入/输出安全分类器
- 策略即代码（允许/拒绝/改写）
- 越狱鲁棒性评估回路


## 1. 分层护栏架构

```
User -> Input Filter -> LLM -> Output Filter -> Tool Firewall -> User
              |                     |                |
           policy DB            safety CLS      allowlist/sandbox
```


In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class SafetyClassifier(nn.Module):
    def __init__(self, vocab=1000, d=64, n_classes=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.fc = nn.Linear(d, n_classes)
        # 0=ok, 1=jailbreak, 2=toxic, 3=pii

    def forward(self, ids):
        x = self.emb(ids).mean(1)
        return self.fc(x)


def rule_pii_scan(text: str) -> bool:
    patterns = [
        r'\b\d{3}-\d{2}-\d{4}\b',  # SSN-like
        r'\b[\w.-]+@[\w.-]+\.\w+\b',
        r'\b1[3-9]\d{9}\b',  # CN phone-like
    ]
    return any(re.search(p, text) for p in patterns)


cls = SafetyClassifier()
# toy train
opt = torch.optim.Adam(cls.parameters(), lr=1e-2)
print('=== Safety Classifier Toy Train ===')
for step in range(50):
    ids = torch.randint(0, 1000, (32, 20))
    labels = torch.randint(0, 4, (32,))
    loss = F.cross_entropy(cls(ids), labels)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 25 == 0 or step == 49:
        print(f'step={step} loss={loss.item():.4f}')

print('PII scan:', rule_pii_scan('contact me at alice@example.com'))
print(f'Key: Combine cheap rules for PII/format with learned classifiers for semantics.')


## 2. 策略引擎：允许 / 拒绝 / 改写

对分类结果映射动作，而不是只返回布尔值。


In [ ]:
ACTIONS = {
    0: 'allow',
    1: 'block',
    2: 'block',
    3: 'redact',
}


def redact(text: str) -> str:
    text = re.sub(r'[\w.-]+@[\w.-]+\.\w+', '[EMAIL]', text)
    text = re.sub(r'\b1[3-9]\d{9}\b', '[PHONE]', text)
    return text


def guardrail_pipeline(text: str, ids: torch.Tensor, model: SafetyClassifier):
    if rule_pii_scan(text):
        return {'action': 'redact', 'output': redact(text), 'reason': 'rule_pii'}
    with torch.no_grad():
        pred = int(model(ids).argmax(-1).item())
    action = ACTIONS[pred]
    if action == 'allow':
        return {'action': 'allow', 'output': text, 'reason': f'cls={pred}'}
    if action == 'redact':
        return {'action': 'redact', 'output': redact(text), 'reason': f'cls={pred}'}
    return {'action': 'block', 'output': '请求已被安全策略拦截。', 'reason': f'cls={pred}'}


samples = [
    ('正常问天气', torch.randint(0, 1000, (1, 16))),
    ('my email is bob@corp.com', torch.randint(0, 1000, (1, 16))),
]
print('=== Policy Engine ===')
for text, ids in samples:
    print(text, '->', guardrail_pipeline(text, ids, cls))
print(f'\nKey: Actionable policies (allow/block/redact/rewrite) are more operable than binary filters.')


## 3. 工具防火墙与评估回路

Agent 场景必须限制工具参数与副作用：路径白名单、SQL 只读、网络 egress 控制。

护栏需像模型一样做回归评测：固定越狱集 + 误拦截集，跟踪 block rate / false positive。


In [ ]:
class ToolFirewall:
    def __init__(self):
        self.allowed = {
            'search': {'q': str},
            'sql_read': {'query': str},
        }
        self.sql_deny = re.compile(r'\b(drop|delete|update|insert|alter)\b', re.I)

    def check(self, tool, args):
        if tool not in self.allowed:
            return False, 'tool_not_allowed'
        schema = self.allowed[tool]
        if set(args) != set(schema):
            return False, 'bad_args'
        if tool == 'sql_read' and self.sql_deny.search(args['query']):
            return False, 'sql_mutation_denied'
        return True, 'ok'


fw = ToolFirewall()
print('=== Tool Firewall ===')
for call in [
    ('search', {'q': 'llm guardrails'}),
    ('sql_read', {'query': 'SELECT * FROM users'}),
    ('sql_read', {'query': 'DROP TABLE users'}),
    ('bash', {'cmd': 'rm -rf /'}),
]:
    print(call, '->', fw.check(*call))

# Simple regression metrics
y_true = [1, 1, 0, 0]  # 1=should_block
y_pred = [1, 0, 0, 1]
tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
prec = tp / max(tp + fp, 1)
rec = tp / max(tp + fn, 1)
print(f'guardrail precision={prec:.2f}, recall={rec:.2f}')
print(f'\nKey: Treat guardrails as a product surface with precision/recall SLOs, not a one-off filter.')


## 课后思考题

1. 护栏误拦截（false positive）过高会怎样伤害产品？如何分层放宽？
2. 模型内对齐与外挂护栏的职责边界如何划分？
3. 对多模态 / Agent 工具调用，护栏要新增哪些控制点？
4. 如何搭建持续的越狱回归集并防止评测集污染？

---
> 本节涵盖了16.5 生产级安全护栏的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
